In [18]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding

In [19]:
# Dados simples: pares inglês-francês
pairs = [
    ("Hello", "Olá"),
    ("Good morning", "Bom dia"),
    ("Good afternoon", "Boa tarde"),
    ("Good evening", "Boa noite"),
    ("Good night", "Boa noite"),
    ("How are you?", "Como você está?"),
    ("I'm fine, thank you.", "Estou bem, obrigado."),
    ("What's your name?", "Qual é o seu nome?"),
    ("My name is John.", "Meu nome é John."),
    ("Nice to meet you.", "Prazer em conhecê-lo."),
    ("Please", "Por favor"),
    ("Thank you", "Obrigado"),
    ("You're welcome", "De nada"),
    ("Excuse me", "Com licença"),
    ("I'm sorry", "Desculpe"),
    ("Yes", "Sim"),
    ("No", "Não"),
    ("Maybe", "Talvez"),
    ("I don't know", "Eu não sei"),
    ("I understand", "Eu entendo"),
    ("I don't understand", "Eu não entendo"),
    ("Can you help me?", "Você pode me ajudar?"),
    ("Where is the bathroom?", "Onde fica o banheiro?"),
    ("How much does it cost?", "Quanto custa?"),
    ("I would like a coffee.", "Eu gostaria de um café."),
    ("Do you speak English?", "Você fala inglês?"),
    ("I speak a little Portuguese.", "Eu falo um pouco de português."),
    ("What time is it?", "Que horas são?"),
    ("I'm hungry.", "Estou com fome."),
    ("I'm thirsty.", "Estou com sede."),
    ("I'm tired.", "Estou cansado."),
    ("I need a doctor.", "Eu preciso de um médico."),
    ("Call the police!", "Chame a polícia!"),
    ("I love you.", "Eu te amo."),
    ("Congratulations!", "Parabéns!"),
    ("Happy birthday!", "Feliz aniversário!"),
    ("Merry Christmas!", "Feliz Natal!"),
    ("Happy New Year!", "Feliz Ano Novo!"),
    ("Good luck!", "Boa sorte!"),
    ("Have a nice day!", "Tenha um bom dia!"),
    ("See you later.", "Até mais."),
    ("Goodbye", "Adeus"),
    ("What's your phone number?", "Qual é o seu número de telefone?"),
    ("Where do you live?", "Onde você mora?"),
    ("I live in São Paulo.", "Eu moro em São Paulo."),
    ("I'm from Brazil.", "Eu sou do Brasil."),
    ("What do you do?", "O que você faz?"),
    ("I'm a teacher.", "Eu sou professor."),
    ("I work in an office.", "Eu trabalho em um escritório."),
    ("I'm a student.", "Eu sou estudante."),
    ("I like to read books.", "Eu gosto de ler livros."),
    ("I enjoy listening to music.", "Eu gosto de ouvir música."),
    ("I like to travel.", "Eu gosto de viajar."),
    ("What's your favorite food?", "Qual é a sua comida favorita?"),
    ("I like pizza.", "Eu gosto de pizza."),
    ("Do you have any pets?", "Você tem animais de estimação?"),
    ("I have a dog.", "Eu tenho um cachorro."),
    ("I don't have any pets.", "Eu não tenho animais de estimação."),
    ("What's your favorite color?", "Qual é a sua cor favorita?"),
    ("My favorite color is blue.", "Minha cor favorita é azul."),
    ("Do you like sports?", "Você gosta de esportes?"),
    ("I like soccer.", "Eu gosto de futebol."),
    ("I don't like sports.", "Eu não gosto de esportes."),
    ("What did you do yesterday?", "O que você fez ontem?"),
    ("I went to the movies.", "Eu fui ao cinema."),
    ("I stayed at home.", "Eu fiquei em casa."),
    ("What are you doing now?", "O que você está fazendo agora?"),
    ("I'm studying English.", "Estou estudando inglês."),
    ("I'm watching TV.", "Estou assistindo TV."),
    ("Do you have brothers or sisters?", "Você tem irmãos ou irmãs?"),
    ("I have one brother.", "Eu tenho um irmão."),
    ("I have two sisters.", "Eu tenho duas irmãs."),
    ("I'm an only child.", "Eu sou filho único."),
    ("What time do you wake up?", "A que horas você acorda?"),
    ("I wake up at 7 a.m.", "Eu acordo às 7 da manhã."),
    ("What time do you go to bed?", "A que horas você vai dormir?"),
    ("I go to bed at 10 p.m.", "Eu vou dormir às 10 da noite."),
    ("Do you like to cook?", "Você gosta de cozinhar?"),
    ("Yes, I love cooking.", "Sim, eu adoro cozinhar."),
    ("No, I don't like to cook.", "Não, eu não gosto de cozinhar."),
    ("What's your favorite movie?", "Qual é o seu filme favorito?"),
    ("I like action movies.", "Eu gosto de filmes de ação."),
    ("I prefer comedies.", "Eu prefiro comédias."),
    ("Do you play any instruments?", "Você toca algum instrumento?"),
    ("I play the guitar.", "Eu toco violão."),
    ("I don't play any instruments.", "Eu não toco nenhum instrumento."),
    ("What's the weather like?", "Como está o tempo?"),
    ("It's sunny.", "Está ensolarado."),
    ("It's raining.", "Está chovendo."),
    ("It's cold.", "Está frio."),
    ("It's hot.", "Está quente."),
    ("Do you like reading?", "Você gosta de ler?"),
    ("Yes, I read every day.", "Sim, eu leio todos os dias."),
    ("No, I prefer watching movies.", "Não, eu prefiro assistir filmes."),
    ("What's your favorite book?", "Qual é o seu livro favorito?"),
    ("I like mystery novels.", "Eu gosto de romances de mistério."),
    ("I enjoy science fiction.", "Eu gosto de ficção científica."),
    ("Can you drive?", "Você sabe dirigir?"),
    ("Yes, I have a driver's license.", "Sim, eu tenho carteira de motorista."),
    ("No, I don't drive.", "Não, eu não dirijo."),
    ("Do you like coffee or tea?", "Você gosta de café ou chá?"),
    ("I prefer coffee.", "Eu prefiro café."),
    ("I like both.", "Eu gosto dos dois."),
    ("I don't like either.", "Eu não gosto de nenhum."),
]



In [20]:
# Separar entradas e saídas
input_texts = [en.lower() for en, _ in pairs]
target_texts = ['<start> ' + fr.lower() + ' <end>' for _, fr in pairs]

In [21]:
# Tokenizar entrada
input_tokenizer = Tokenizer(filters='')
input_tokenizer.fit_on_texts(input_texts)
input_sequences = input_tokenizer.texts_to_sequences(input_texts)
input_word_index = input_tokenizer.word_index
num_encoder_tokens = len(input_word_index) + 1
max_encoder_seq_length = max(len(seq) for seq in input_sequences)

In [22]:
# Tokenizar saída
target_tokenizer = Tokenizer(filters='')
target_tokenizer.fit_on_texts(target_texts)
target_sequences = target_tokenizer.texts_to_sequences(target_texts)
target_word_index = target_tokenizer.word_index
num_decoder_tokens = len(target_word_index) + 1
max_decoder_seq_length = max(len(seq) for seq in target_sequences)

In [23]:
# Padding
encoder_input_data = pad_sequences(input_sequences, maxlen=max_encoder_seq_length, padding='post')
decoder_input_data = pad_sequences([seq[:-1] for seq in target_sequences], maxlen=max_decoder_seq_length-1, padding='post')
decoder_target_data = pad_sequences([seq[1:] for seq in target_sequences], maxlen=max_decoder_seq_length-1, padding='post')


In [24]:
# One-hot para target
decoder_target_onehot = tf.keras.utils.to_categorical(decoder_target_data, num_classes=num_decoder_tokens)


In [25]:
# Hiperparâmetros
latent_dim = 256
embedding_dim = 100

In [26]:
# Encoder
encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(input_dim=num_encoder_tokens, output_dim=embedding_dim)(encoder_inputs)
encoder_lstm, state_h, state_c = LSTM(latent_dim, return_state=True)(enc_emb)
encoder_states = [state_h, state_c]

In [27]:
# Decoder
decoder_inputs = Input(shape=(None,))
dec_emb_layer = Embedding(input_dim=num_decoder_tokens, output_dim=embedding_dim)
dec_emb = dec_emb_layer(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = Dense(num_decoder_tokens, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

In [28]:
# Modelo
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])

In [29]:
# Treinar
model.fit([encoder_input_data, decoder_input_data], decoder_target_onehot,
          batch_size=2,
          epochs=100,
          validation_split=0.2)

Epoch 1/100
42/42 ━━━━━━━━━━━━━━━━━━━━ 8s 35ms/step - accuracy: 0.3839 - loss: 3.9356 - val_accuracy: 0.3571 - val_loss: 3.0537
Epoch 2/100
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.5090 - loss: 2.5374 - val_accuracy: 0.5476 - val_loss: 3.1302
Epoch 3/100
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.6020 - loss: 2.3058 - val_accuracy: 0.5476 - val_loss: 2.8129
Epoch 4/100
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.6011 - loss: 2.1571 - val_accuracy: 0.5238 - val_loss: 2.8944
Epoch 5/100
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.5792 - loss: 2.1649 - val_accuracy: 0.5476 - val_loss: 2.8022
Epoch 6/100
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.5639 - loss: 2.1373 - val_accuracy: 0.5595 - val_loss: 2.8225
Epoch 7/100
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.6123 - loss: 1.9048 - val_accuracy: 0.5893 - val_loss: 2.7165
Epoch 8/100
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.6000 - loss: 1.8995 - val_accuracy: 0.

In [30]:
encoder_model = Model(encoder_inputs, encoder_states)

In [31]:
# Entradas do estado anterior (hidden e cell)
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

# Entrada do token atual
dec_emb2 = dec_emb_layer(decoder_inputs)

# Passa pelo LSTM com estado anterior
decoder_outputs2, state_h2, state_c2 = decoder_lstm(dec_emb2, initial_state=decoder_states_inputs)
decoder_states2 = [state_h2, state_c2]

# Passa pela densa (softmax)
decoder_outputs2 = decoder_dense(decoder_outputs2)

# Modelo de decoder (1 passo por vez)
decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs2] + decoder_states2
)

In [32]:
reverse_target_word_index = dict((i, word) for word, i in target_word_index.items())

def decode_sequence(input_seq):
    # Obtém o estado inicial do encoder
    states_value = encoder_model.predict(input_seq)

    # Começa com o token <start>
    target_seq = np.array([[target_word_index['<start>']]])

    stop_condition = False
    decoded_sentence = ''

    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)

        # Escolhe o token com maior probabilidade
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = reverse_target_word_index.get(sampled_token_index, '')

        if sampled_word == '<end>' or len(decoded_sentence.split()) > max_decoder_seq_length:
            stop_condition = True
        else:
            decoded_sentence += ' ' + sampled_word

        # Atualiza a entrada e os estados
        target_seq = np.array([[sampled_token_index]])
        states_value = [h, c]

    return decoded_sentence.strip()


In [33]:
def translate(sentence):
    # Pré-processa: tokenize e pad
    seq = input_tokenizer.texts_to_sequences([sentence.lower()])
    seq = pad_sequences(seq, maxlen=max_encoder_seq_length, padding='post')
    return decode_sequence(seq)

# Exemplo de uso
print("Inglês: hi")
print("Portugues (gerado):", translate("hi"))

print("Inglês: I read every day")
print("Portugues (gerado):", translate("I read every day"))


print("Inglês: You stayed at home")
print("Portugues (gerado):", translate("Inglês: You stayed at home"))


Inglês: hi
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 335ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 316ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 307ms/step
Portugues (gerado): sim
Inglês: I read every day
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Portugues (gerado): eu gosto de pizza.
Inglês: You stayed at home
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Portugues (gerado): prazer em conhecê-lo.
